## Check that Boto3 is installed

In [13]:
import boto3
import pandas as pd

In [2]:
print("Boto3 version:", boto3.__version__)
print("Pandas version:", pd.__version__)

Boto3 version: 1.43.46
Pandas version: 2.3.3


In [ ]:
# If Boto3 isn't installed:
# run: %pip install boto3

## Test AWS authentication

In [3]:
sts = boto3.client("sts")

identity = sts.get_caller_identity()

identity

{'UserId': 'AROA4NFHWSK24RJH6PJAG:SageMaker',
 'Account': '852902711989',
 'Arn': 'arn:aws:sts::852902711989:assumed-role/AmazonSageMaker-ExecutionRole-20260811T192382/SageMaker',
 'ResponseMetadata': {'RequestId': '92071719-a287-46c6-a28d-2c609692c066',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '92071719-a287-46c6-a28d-2c609692c066',
   'x-amz-sts-extended-request-id': 'MTpldS1jZW50cmFsLTE6UzoxNzg2ODEyMzM1MDg1OlI6V0pkcTB1TXU=',
   'content-type': 'text/xml',
   'content-length': '470',
   'date': 'Sat, 15 Aug 2026 16:45:35 GMT'},
  'RetryAttempts': 0}}

**Why are we doing this?**

*This confirms that*:

    SageMaker JupyterLab
    
            ↓
    AWS Credentials / IAM Role
    
            ↓
    AWS Account

*is working*. You don't need to manually put an access key or secret key into your Python code.

## Test S3 access independently

Before using your DataLoader, let's test S3 itself. Replace the values with your actual bucket and object key:

In [4]:
s3 = boto3.client("s3")

bucket_name = "sira-data"
object_key = "raw/incident_reports_1000.csv"

response = s3.head_object(
    Bucket=bucket_name,
    Key=object_key
)

print("File exists!")
print("File size:", response["ContentLength"], "bytes")

File exists!
File size: 126204 bytes


If it's `error's out`, then you need to give permissions. Let's catch the error using `try-catch`.

In [5]:
# import boto3
# from botocore.exceptions import ClientError

# s3 = boto3.client('s3') 
# bucket_name = 'sira-data' 
# object_key = 'raw/incident_reports_1000.csv' 

# try:
#     response = s3.head_object(Bucket=bucket_name, Key=object_key) 
#     print("File exists!") 
#     print("File size:", response['ContentLength'], "bytes")
# except ClientError as e:
#     error_code = e.response['Error']['Code']
#     if error_code == '403':
#         print("Access Denied: Check your IAM permissions or bucket policy.")
#     elif error_code == '404':
#         print("File Not Found: Check your bucket name and object key.")
#     else:
#         print(f"An unexpected error occurred: {e}")


Access Denied: Check your IAM permissions or bucket policy.


Since the `403` Forbidden error persists, you need to find out exactly who you are logged in as and check if that identity has the required access.

In [6]:
# import boto3

# sts = boto3.client('sts')
# try:
#     identity = sts.get_caller_identity()
#     print("Logged in as identity:")
#     print(f"  - Account: {identity['Account']}")
#     print(f"  - ARN: {identity['Arn']}")
# except Exception as e:
#     print(f"Could not fetch identity: {e}")


Logged in as identity:
  - Account: 852902711989
  - ARN: arn:aws:sts::852902711989:assumed-role/AmazonSageMaker-ExecutionRole-20260811T192382/SageMaker


### To fix the `403` Forbidden error, you must attach S3 permissions directly to the specific SageMaker Execution Role in the AWS Console.

**Follow the steps below to give role permission**

1. Locate the Role in IAM
   * Open the AWS Management Console
   * Navigate to the IAM (Identity and Access Management) service
   * Click on Roles in the left sidebar.
   * Search for the name of your role: e.g `AmazonSageMaker-ExecutionRole-20260811T192382` and click on it
     
2. Attach S3 Access Permissions
   Once you are looking at the summary page for that specific role:

   * Under the Permissions tab, click Add permissions -> Attach policies
   * Search for `AmazonS3ReadOnlyAccess` (or `AmazonS3FullAccess` if you plan to write data back to S3 later)
   * Check the box next to the policy and click Add permissions
    
3. Check Cross-Account Access (If applicable)
   Does the S3 bucket sira-data belong to a different AWS Account than 852902711989?

   If yes, giving permissions to SageMaker isn't enough. The owner of the external bucket must update their Bucket Policy to explicitly allow your SageMaker role:
```json
   {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "AWS": "arn:aws:iam::852902711989:role/service-role/AmazonSageMaker-ExecutionRole-20260811T192382"
            },
            "Action": [
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                "arn:aws:s3:::sira-data",
                "arn:aws:s3:::sira-data/*"
            ]
        }
    ]
}```


## List the files in your S3 folder

In [5]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix="raw/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])

raw/
raw/incident_reports_1000.csv


You should see something like: `raw/incident_reports_1000.csv`. This is useful because it confirms that your IAM role can list objects, not just access one known object.

## Test downloading the CSV with Boto3

In [6]:
response = s3.get_object(
    Bucket=bucket_name,
    Key=object_key
)

df = pd.read_csv(response["Body"])

df.head()

,incident_id,report_text,location,reported_by,department,severity,incident_type,report_date,shift,status
0,1,Pressure leak detected on Pipeline 7 during ro...,Platform B,Aisha,Instrumentation,High,Leak,2025-09-30,Day,Open
1,2,NaN,Compressor Station,Mary,Utilities,Low,Inspection,2025-12-15,Day,Closed
2,3,NaN,Tank Farm,Mary,Utilities,Medium,Slip/Fall,2025-09-09,Day,Open
3,4,Smoke observed from electrical control panel.,Refinery,Joy,Production,High,Electrical,2026-01-22,Day,Open
4,5,NaN,Gas Processing Unit,Samuel,Production,Medium,Equipment Failure,2025-02-14,Day,Closed


## Check the DataFrame

In [7]:
print("Shape:", df.shape)

Shape: (1000, 10)


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   incident_id    1000 non-null   int64 
 1   report_text    962 non-null    object
 2   location       979 non-null    object
 3   reported_by    974 non-null    object
 4   department     1000 non-null   object
 5   severity       1000 non-null   object
 6   incident_type  1000 non-null   object
 7   report_date    1000 non-null   object
 8   shift          1000 non-null   object
 9   status         1000 non-null   object
dtypes: int64(1), object(9)
memory usage: 78.3+ KB


In [9]:
df.columns

Index(['incident_id', 'report_text', 'location', 'reported_by', 'department',
       'severity', 'incident_type', 'report_date', 'shift', 'status'],
      dtype='object')

In [10]:
df.tail()

,incident_id,report_text,location,reported_by,department,severity,incident_type,report_date,shift,status
995,683,Small hydraulic fluid leak found beneath crane.,Manifold,David,Electrical,MEDIUM,Leak,2026-06-13,Day,Closed
996,582,small hydraulic fluid leak found beneath crane.,Pipeline 8,John,Electrical,Medium,Leak,Apr 02 2026,Night,Under Investigation
997,351,Routine valve maintenance completed without is...,Platform B,Grace,Mechanical,L,Maintenance,2025-12-12,Night,Open
998,366,Unauthorized personnel entered restricted area.,Pipeline 12,Blessing,Electrical,High,Security,2026-01-06,Day,Under Investigation
999,145,Corrosion observed on external pipeline coating.,Pipeline 7,Fatima,Inspection,Medium,Corrosion,27/03/2025,Day,Resolved


## Verify the expected columns

In [11]:
expected_columns = [
    "incident_id",
    "report_date",
    "location",
    "department",
    "severity",
    "status",
    "shift",
    "report_text"
]

missing_columns = [
    column
    for column in expected_columns
    if column not in df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All expected columns are present.")

All expected columns are present.


This is an important engineering check. You're not simply asking:

> "Did Pandas load the CSV?"

You're asking:

> "Did the correct dataset load with the schema my application expects?"

## Test your actual `DataLoader`

**First, check where your notebook is running.**

In [18]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
/home/sagemaker-user/sira-dev/notebooks


In [19]:
# Check your project structure
for path in Path.cwd().iterdir():
    print(path)

/home/sagemaker-user/sira-dev/notebooks/.ipynb_checkpoints
/home/sagemaker-user/sira-dev/notebooks/01_python_to_ml_engineering.ipynb
/home/sagemaker-user/sira-dev/notebooks/data_loading_concfirmation.ipynb


In [20]:
import sys

# Current directory:
# /home/sagemaker-user/sira-dev/notebooks
current_dir = Path.cwd()

# Move one level up to the project root
project_root = current_dir.parent

# Add project root to Python's module search path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("src directory exists:", (project_root / "src").exists())

Project root: /home/sagemaker-user/sira-dev
src directory exists: True


In [24]:
# print(list(project_root.iterdir()))
# print(list((project_root / "src").iterdir()))

## Update your `DataLoader` with the script below.

```python
"""
Data Loader Module

Responsible for loading datasets from local storage
or Amazon S3.
"""

from pathlib import Path
from io import BytesIO
import logging

import boto3
import pandas as pd


logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)


class DataLoader:
    """
    Responsible for loading datasets.

    This class supports loading CSV files from:

    1. Local project storage
    2. Amazon S3

    The rest of the application does not need to know
    where the dataset is stored.
    """

    def __init__(
        self,
        source="local",
        s3_bucket=None,
        s3_key=None
    ):
        """
        Initialize the DataLoader.

        Parameters
        ----------
        source : str
            Data source.

            Options:
                "local"
                "s3"

        s3_bucket : str, optional
            Name of the S3 bucket.

        s3_key : str, optional
            Object key/path inside the S3 bucket.
        """

        self.source = source.lower()

        # LOCAL DATA CONFIGURATION
        project_root = (
            Path(__file__)
            .resolve()
            .parent
            .parent
        )

        self.local_path = (
            project_root
            / "data"
            / "raw"
            / "incident_reports_1000.csv"
        )

        # AWS S3 CONFIGURATION
        self.s3_bucket = s3_bucket
        self.s3_key = s3_key

        # Create the S3 client only when needed
        if self.source == "s3":

            self.s3_client = boto3.client("s3")

    # LOAD DATA
    def load_data(self):
        """
        Load the dataset based on the configured source.
        """

        if self.source == "local":
            return self._load_from_local()
        elif self.source == "s3":
            return self._load_from_s3()
        else:
            raise ValueError(
                "Invalid source. Use 'local' or 's3'."
            )

    # LOCAL LOADING
    def _load_from_local(self):
        """
        Load CSV from local project storage.
        """

        logging.info(
            f"Loading local dataset: {self.local_path}"
        )

        if not self.local_path.exists():
            raise FileNotFoundError(
                f"Dataset not found: {self.local_path}"
            )

        df = pd.read_csv(
            self.local_path,
            index_col=False
        )

        logging.info(
            f"Data loaded successfully: {df.shape}"
        )

        return df

    # S3 LOADING
    def _load_from_s3(self):
        """
        Load CSV directly from Amazon S3.
        """

        if not self.s3_bucket:
            raise ValueError(
                "s3_bucket must be provided when "
                "source='s3'."
            )

        if not self.s3_key:

            raise ValueError(
                "s3_key must be provided when "
                "source='s3'."
            )

        logging.info(
            f"Loading dataset from S3: "
            f"s3://{self.s3_bucket}/{self.s3_key}"
        )

        response = self.s3_client.get_object(
            Bucket=self.s3_bucket,
            Key=self.s3_key
        )

        df = pd.read_csv(
            BytesIO(response["Body"].read()),
            index_col=False
        )

        logging.info(
            f"Data loaded successfully: {df.shape}"
        )

        return df

```

In [26]:
from src.data_loader import DataLoader

print("DataLoader imported successfully!")

DataLoader imported successfully!


In [27]:
loader = DataLoader(
    source="s3",
    s3_bucket="sira-data",
    s3_key="raw/incident_reports_1000.csv"
)

df = loader.load_data()

print(df.shape)

df.head()

INFO: Loading dataset from S3: s3://sira-data/raw/incident_reports_1000.csv


INFO: Data loaded successfully: (1000, 10)


(1000, 10)


,incident_id,report_text,location,reported_by,department,severity,incident_type,report_date,shift,status
0,1,Pressure leak detected on Pipeline 7 during ro...,Platform B,Aisha,Instrumentation,High,Leak,2025-09-30,Day,Open
1,2,NaN,Compressor Station,Mary,Utilities,Low,Inspection,2025-12-15,Day,Closed
2,3,NaN,Tank Farm,Mary,Utilities,Medium,Slip/Fall,2025-09-09,Day,Open
3,4,Smoke observed from electrical control panel.,Refinery,Joy,Production,High,Electrical,2026-01-22,Day,Open
4,5,NaN,Gas Processing Unit,Samuel,Production,Medium,Equipment Failure,2025-02-14,Day,Closed


The `output` above tells us now that the following are all working:

- ✅ SageMaker JupyterLab
- ✅ Python environment
- ✅ Boto3
- ✅ IAM authentication
- ✅ S3 access
- ✅ Correct bucket: `sira-data`
- ✅ Correct object key: `raw/incident_reports_1000.csv`
- ✅ Your DataLoader module
- ✅ Pandas
- ✅ Dataset retrieval
- ✅ DataFrame creation

In [28]:
# Inspect the columns
print("Columns:")
for column in df.columns:
    print(f"- {column}")

Columns:
- incident_id
- report_text
- location
- reported_by
- department
- severity
- incident_type
- report_date
- shift
- status


In [29]:
# Inspect the first records
df.head()

,incident_id,report_text,location,reported_by,department,severity,incident_type,report_date,shift,status
0,1,Pressure leak detected on Pipeline 7 during ro...,Platform B,Aisha,Instrumentation,High,Leak,2025-09-30,Day,Open
1,2,NaN,Compressor Station,Mary,Utilities,Low,Inspection,2025-12-15,Day,Closed
2,3,NaN,Tank Farm,Mary,Utilities,Medium,Slip/Fall,2025-09-09,Day,Open
3,4,Smoke observed from electrical control panel.,Refinery,Joy,Production,High,Electrical,2026-01-22,Day,Open
4,5,NaN,Gas Processing Unit,Samuel,Production,Medium,Equipment Failure,2025-02-14,Day,Closed


In [30]:
# Inspect the data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   incident_id    1000 non-null   int64 
 1   report_text    962 non-null    object
 2   location       979 non-null    object
 3   reported_by    974 non-null    object
 4   department     1000 non-null   object
 5   severity       1000 non-null   object
 6   incident_type  1000 non-null   object
 7   report_date    1000 non-null   object
 8   shift          1000 non-null   object
 9   status         1000 non-null   object
dtypes: int64(1), object(9)
memory usage: 78.3+ KB


In [31]:
# Check the dimensions
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 1000
Number of columns: 10


In [32]:
# Check missing values
df.isnull().sum()

incident_id       0
report_text      38
location         21
reported_by      26
department        0
severity          0
incident_type     0
report_date       0
shift             0
status            0
dtype: int64

This is particularly useful because we're going to pass this dataset into your preprocessing.py module next.

In [33]:
# Check duplicates
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 40


In [34]:
# Validate the SIRA schema
assert "report_text" in df.columns

print("SIRA schema validation passed.")

SIRA schema validation passed.


## Final validation cell

In [35]:
assert df is not None
assert len(df) == 1000
assert len(df.columns) == 10
assert "report_text" in df.columns

print("====================================")
print("SIRA CLOUD DATA LOADING TEST PASSED")
print("====================================")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("Source: Amazon S3")
print("Bucket: sira-data")
print("Object: raw/incident_reports_1000.csv")

SIRA CLOUD DATA LOADING TEST PASSED
Rows: 1000
Columns: 10
Source: Amazon S3
Bucket: sira-data
Object: raw/incident_reports_1000.csv


**We now have a clear `milestone`.**

                    SIRA CLOUD PIPELINE
                           │
                           ▼
                    ┌─────────────┐
                    │  Amazon S3  │
                    │  sira-data  │
                    └──────┬──────┘
                           │
                           ▼
                    ┌─────────────┐
                    │ DataLoader  │
                    │  + Boto3    │
                    └──────┬──────┘
                           │
                           ▼
                    ┌─────────────┐
                    │   Pandas    │
                    │ DataFrame   │
                    └──────┬──────┘
                           │
                           ▼
                    1000 × 10

This is more significant than simply "the CSV loaded." We've now decoupled our SIRA application from local data storage. Our `DataLoader` can retrieve the dataset from cloud storage without the CSV having to exist in the project's data/ directory.